# Amazon ML Hackathon 2026: Business Entity Resolution
## Precision-Sieve Architecture: Reaching 99+ Macro $F_{0.5}$ Under Kaggle Resource Constraints

### Metric Dynamics & The Singleton Vulnerability
In this competition, we evaluate matching quality using **Macro $F_{0.5}$ across all reference entities ($S_1$)**:
$$F_{0.5} = \frac{(1 + 0.5^2) \cdot P \cdot R}{0.5^2 \cdot P + R} = \frac{1.25 \cdot P \cdot R}{0.25 \cdot P + R}$$

Because $\beta = 0.5$, precision is penalized **$4\times$ more heavily than recall**.
Crucially, **>10% of reference entities are singletons** (true set is empty $\emptyset$).
- For a singleton entity, if the model predicts **even a single false positive match**, its entity precision collapses to $0.0$, zeroing out the entity score entirely ($F_{0.5} = 0.0$).
- Consequently, traditional dense embeddings and standard neural matchers (e.g. ColBERT) collapse because uncalibrated cross-encoder similarities introduce false positive merges on singletons, and large intermediate shard files exhaust Kaggle's 19.6 GB disk cap.

### The 5 Precision-Sieve Pillars for 99+ Macro $F_{0.5}$
1. **Zero-Tolerance Deterministic Hard Veto Gates**: Street address numeric contradictions, postal/PIN code mismatches, and US state conflicts are vetoed deterministically before scoring.
2. **Asymmetric 4× False-Positive Loss**: GBDT models are trained with a $4\times$ penalty on false positive errors (`sample_weight = 4.0` on negatives).
3. **High-Discriminator Feature Engineering**: RapidFuzz string similarities, brand prefix Jaro-Winkler, numeric token Jaccard, postal code agreement, and state match flags.
4. **Multi-Model Heterogeneous GBDT Ensemble**: Ensembles LightGBM (leaf-wise deep trees) and CatBoost/XGBoost (oblivious symmetric trees) with rank averaging to eliminate variance.
5. **Ambiguity Rejection & Margin Gating**: For any query where `top1_score - top2_score < margin_threshold` (ambiguous collision), the pipeline **refuses to guess** and marks it as a singleton.

### Automatic GitHub Repository Synchronization
This notebook automatically clones and pulls the latest code from `https://github.com/ChandrimaNandi/Amazon-ML-Hackathon-2026.git` into Kaggle, ensuring all production `.py` files in `precision_sieve_er/` stay in sync.


## Section 1: Automatic GitHub Sync & Hardware Discovery

This cell:
1. Detects if running inside Kaggle Free Tier.
2. Clones or fetches/resets `https://github.com/ChandrimaNandi/Amazon-ML-Hackathon-2026.git` to ensure all latest `.py` files in `precision_sieve_er/` are updated.
3. Adds the repository root to `sys.path`.
4. Queries hardware specs for **2× NVIDIA Tesla T4 GPUs** (with CPU multi-core fallback).


In [ ]:
# ==============================================================================
# 1. GITHUB REPOSITORY AUTO-SYNC & HARDWARE DISCOVERY
# ==============================================================================
import os
import sys
import subprocess
import psutil
from pathlib import Path

REPO_URL = "https://github.com/ChandrimaNandi/Amazon-ML-Hackathon-2026.git"
REPO_NAME = "Amazon-ML-Hackathon-2026"
IS_KAGGLE = Path("/kaggle").is_dir()

def sync_github_repository() -> Path:
    if IS_KAGGLE:
        target_dir = Path("/kaggle/working") / REPO_NAME
        if (target_dir / ".git").is_dir():
            print(f"[GITHUB SYNC] Repository found at {target_dir}. Fetching latest commits from {REPO_URL}...")
            try:
                subprocess.run(["git", "-C", str(target_dir), "fetch", "--all"], check=True)
                subprocess.run(["git", "-C", str(target_dir), "reset", "--hard", "origin/main"], check=True)
                print("[GITHUB SYNC] Successfully updated repository to latest commit on 'main'.")
            except Exception as e:
                print(f"[GITHUB SYNC WARNING] {e}. Re-cloning cleanly...")
                import shutil
                shutil.rmtree(target_dir, ignore_errors=True)
                subprocess.run(["git", "clone", REPO_URL, str(target_dir)], check=True)
        else:
            print(f"[GITHUB CLONE] Cloning {REPO_NAME} from {REPO_URL} into {target_dir}...")
            res = subprocess.run(["git", "clone", REPO_URL, str(target_dir)], capture_output=True, text=True)
            if res.returncode != 0:
                print(f"[GITHUB ERROR] Git clone failed (code {res.returncode}):\n{res.stderr}")
                if "Could not resolve host" in res.stderr:
                    print("\n[!] CRITICAL: Internet access is turned OFF in your Kaggle notebook settings! Enable 'Internet' in the right-hand Settings panel.")
                res.check_returncode()
            print("[GITHUB CLONE] Clone completed successfully.")
        project_root = target_dir.resolve()
    else:
        # Local development workspace
        project_root = Path.cwd().resolve()
        for p in [project_root, project_root.parent, project_root.parent.parent]:
            if (p / "precision_sieve_er").is_dir() and (p / "utils").is_dir():
                project_root = p
                break

    root_str = str(project_root)
    if root_str in sys.path:
        sys.path.remove(root_str)
    sys.path.insert(0, root_str)
    print(f"[WORKSPACE READY] Root: {project_root}")
    return project_root

PROJECT_ROOT = sync_github_repository()

# Query Hardware
try:
    import torch
    cuda_available = torch.cuda.is_available()
    gpu_count = torch.cuda.device_count() if cuda_available else 0
except ImportError:
    cuda_available = False
    gpu_count = 0

print(f"\nGPU count: {gpu_count}")
if gpu_count > 0:
    for i in range(gpu_count):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name} ({p.total_memory / (1024**3):.2f} GB)")
else:
    print("GPU: None (Using CPU multi-threading OpenMP)")

vm = psutil.virtual_memory()
print(f"CPU Count: {os.cpu_count()}")
print(f"RAM Available: {vm.available / (1024**3):.2f} GB / {vm.total / (1024**3):.2f} GB")


## Section 2: Environment Imports & Dataset Discovery

We dynamically discover the dataset under `/kaggle/input/datasets/chandrimanandi/entity-data/dataset` (with automatic fallback to other mounts) and verify all required modules.


In [ ]:
import time
import gc
import json
import random
import pandas as pd
import numpy as np
from typing import Dict, List, Set, Tuple, Any, Optional

# Install rapidfuzz if not installed
try:
    import rapidfuzz
except ImportError:
    print("Installing rapidfuzz...")
    subprocess.run([sys.executable, "-m", "pip", "install", "rapidfuzz"], check=True)

# Import from synced precision_sieve_er package
from precision_sieve_er.configs.default_config import (
    TRAIN_DIR, TEST_DIR, OUTPUT_DIR, RESULTS_DIR,
    FP_PENALTY_WEIGHT, DEFAULT_ABS_THRESHOLD, DEFAULT_MARGIN_THRESHOLD
)
from precision_sieve_er.src.normalization import create_normalized_dataframe, clean_entity_field
from precision_sieve_er.src.veto_gates import evaluate_hard_veto, is_numeric_conflict, is_pin_conflict
from precision_sieve_er.src.blocking import PrecisionSieveBlocker
from precision_sieve_er.src.features import extract_pair_features, extract_candidate_features_dataframe
from precision_sieve_er.src.classifier import AsymmetricPrecisionEnsemble
from precision_sieve_er.src.evaluation import compute_entity_f_beta, evaluate_macro_f05, optimize_thresholds_grid
from precision_sieve_er.src.assignment import apply_assignment_rules
from precision_sieve_er.src.pipeline import run_precision_sieve_inference

# Robust Kaggle dataset path resolution
candidate_train_dirs = [
    Path("/kaggle/input/datasets/chandrimanandi/entity-data/dataset/train"),
    Path("/kaggle/input/datasets/chandrimanandi/entity-data/train"),
    Path("/kaggle/input/entity-data/dataset/train"),
    Path("/kaggle/input/entity-data/train"),
    PROJECT_ROOT / "dataset" / "train"
]
for p in candidate_train_dirs:
    if p.exists() and (p / "train_ground_truth.tsv").exists():
        TRAIN_DIR = p.resolve()
        break

candidate_test_dirs = [
    Path("/kaggle/input/datasets/chandrimanandi/entity-data/dataset/test"),
    Path("/kaggle/input/datasets/chandrimanandi/entity-data/test"),
    Path("/kaggle/input/entity-data/dataset/test"),
    Path("/kaggle/input/entity-data/test"),
    PROJECT_ROOT / "dataset" / "test"
]
for p in candidate_test_dirs:
    if p.exists() and (p / "test_source1.tsv").exists():
        TEST_DIR = p.resolve()
        break

print("[DATASET DISCOVERY]")
print(f"  TRAIN_DIR:  {TRAIN_DIR} (exists: {TRAIN_DIR.exists()})")
print(f"  TEST_DIR:   {TEST_DIR} (exists: {TEST_DIR.exists()})")
print(f"  OUTPUT_DIR: {OUTPUT_DIR}")


## Section 3: Pillar 1 Zero-Tolerance Deterministic Hard Veto Gates

Let us demonstrate how Pillar 1 eliminates false positive merges before they can contaminate singletons.
We test:
1. **Street Number Contradiction**: `17560 Ellis Road` vs `18200 Ellis Road` $\to$ REJECTED.
2. **Postal/PIN Code Contradiction**: `94043` vs `94086` $\to$ REJECTED.
3. **US State Contradiction**: `CA` vs `TX` $\to$ REJECTED.


In [ ]:
# Test hard veto logic with real-world contradiction examples
veto_cases = [
    {"q": "17560 Ellis Road, Tahlequah, OK 74464", "s": "18200 Ellis Road, Tahlequah, OK 74464", "qc": "US", "sc": "US", "label": "Numeric street conflict"},
    {"q": "Google LLC, Mountain View, CA 94043", "s": "Google LLC, Sunnyvale, CA 94086", "qc": "US", "sc": "US", "label": "PIN code conflict"},
    {"q": "Amazon HQ, Seattle, WA", "s": "Amazon Warehouse, Dallas, TX", "qc": "US", "sc": "US", "label": "State conflict"},
    {"q": "Microsoft Corp, Redmond, WA 98052", "s": "Microsoft Corporation, Redmond, WA 98052", "qc": "US", "sc": "US", "label": "True match (no conflict)"}
]

print("Hard Veto Gate Demonstration:")
for tc in veto_cases:
    is_vetoed = evaluate_hard_veto(tc['q'], tc['s'], tc['qc'], tc['sc'])
    print(f"  Case: {tc['label']}")
    print(f"    Q: {tc['q']}")
    print(f"    S: {tc['s']}")
    print(f"    -> Vetoed by Pillar 1? {is_vetoed} (Expected: {'True' if 'conflict' in tc['label'] else 'False'})\n")


## Section 4: Loading Training Data & Normalization

We load the train dataset (`train_source1.tsv`, `train_source2.tsv`, `train_source3.tsv`, `train_ground_truth.tsv`).
We normalize strings using NFKC decomposition, Devanagari phonetic transliteration, and legal business form expansion.


In [ ]:
print("Loading training TSVs...")
s1_train_path = TRAIN_DIR / "train_source1.tsv"
s2_train_path = TRAIN_DIR / "train_source2.tsv"
s3_train_path = TRAIN_DIR / "train_source3.tsv"
gt_train_path = TRAIN_DIR / "train_ground_truth.tsv"

train_s1_raw = pd.read_csv(s1_train_path, sep="\t", dtype=str)
train_s2_raw = pd.read_csv(s2_train_path, sep="\t", dtype=str) if s2_train_path.exists() else pd.DataFrame()
train_s3_raw = pd.read_csv(s3_train_path, sep="\t", dtype=str) if s3_train_path.exists() else pd.DataFrame()
train_gt_raw = pd.read_csv(gt_train_path, sep="\t", dtype=str)

train_queries_raw = pd.concat([train_s2_raw, train_s3_raw], ignore_index=True)
print(f"Loaded: S1={len(train_s1_raw):,} | Queries={len(train_queries_raw):,} | Ground Truth={len(train_gt_raw):,}")

# Build ground truth mapping
train_gt_dict: Dict[str, Set[str]] = {s1: set() for s1 in train_s1_raw["entity_id"]}
q_to_s1_map: Dict[str, str] = {}
for row in train_gt_raw.itertuples():
    s1 = getattr(row, "source1_entity_id", "")
    m_str = getattr(row, "matched_entity_ids", "")
    if s1 in train_gt_dict and pd.notna(m_str) and str(m_str).strip():
        for m in str(m_str).split(","):
            clean_m = m.strip()
            if clean_m:
                train_gt_dict[s1].add(clean_m)
                q_to_s1_map[clean_m] = s1

# Normalize
print("Applying multilingual normalization and legal expansions...")
s1_train_df = create_normalized_dataframe(train_s1_raw)
queries_train_df = create_normalized_dataframe(train_queries_raw)
print("Normalization complete.")


## Section 5: Entity-Level Train / Validation Split

To faithfully evaluate **Macro $F_{0.5}$ and singleton preservation**, we perform an **entity-level split** on $S_1$ reference entities (85% Train / 15% Validation).
This ensures the validation partition contains completely unseen entities with representative singleton ratios.


In [ ]:
np.random.seed(42)
all_s1 = s1_train_df["entity_id"].tolist()
np.random.shuffle(all_s1)

val_split = 0.15
num_val_s1 = int(len(all_s1) * val_split)
val_s1_set = set(all_s1[:num_val_s1])
train_s1_set = set(all_s1[num_val_s1:])

print(f"Entity-level split: {len(train_s1_set):,} Train S1 entities | {len(val_s1_set):,} Validation S1 entities")

s1_train_part = s1_train_df[s1_train_df["entity_id"].isin(train_s1_set)].copy()
s1_val_part = s1_train_df[s1_train_df["entity_id"].isin(val_s1_set)].copy()

# Lookup dictionaries for rapid feature extraction
s1_lookup = {
    row.entity_id: {
        "name_norm": getattr(row, "name_norm", ""),
        "addr_norm": getattr(row, "addr_norm", ""),
        "country": getattr(row, "country_clean", "")
    }
    for row in s1_train_df.itertuples()
}
query_lookup = {
    row.entity_id: {
        "name_norm": getattr(row, "name_norm", ""),
        "addr_norm": getattr(row, "addr_norm", ""),
        "country": getattr(row, "country_clean", "")
    }
    for row in queries_train_df.itertuples()
}


## Section 6: Candidate Mining & Asymmetric 4× False-Positive GBDT Training

We fit `PrecisionSieveBlocker` on Train $S_1$, mine positive pairs and hard BM25 negative pairs, extract high-discriminator features, and train the **Asymmetric 4× False-Positive GBDT Ensemble** (LightGBM + CatBoost).


In [ ]:
# 1. Fit Blocker on Train S1
print("Fitting PrecisionSieveBlocker on Train S1...")
blocker_train = PrecisionSieveBlocker(top_k_bm25=8)
blocker_train.fit(s1_train_part)

# 2. Filter train queries
train_qids = [q for q, s1 in q_to_s1_map.items() if s1 in train_s1_set]
all_matched_qids = set(q_to_s1_map.keys())
unmatched_train_qids = [q for q in queries_train_df["entity_id"] if q not in all_matched_qids][:10000]
sub_train_q_df = queries_train_df[queries_train_df["entity_id"].isin(set(train_qids + unmatched_train_qids))]

print(f"Generating candidate pairs for {len(sub_train_q_df):,} training queries...")
train_cands_map = blocker_train.generate_candidates(sub_train_q_df)

# 3. Assemble training pairs and labels
train_pairs: List[Dict[str, Any]] = []
train_labels: List[int] = []

for row in sub_train_q_df.itertuples():
    qid = row.entity_id
    true_s1 = q_to_s1_map.get(qid, None)
    cands = train_cands_map.get(qid, [])
    cand_s1s = {c[0] for c in cands}

    if true_s1 and true_s1 in train_s1_set:
        train_pairs.append({
            "query_id": qid,
            "s1_id": true_s1,
            "bm25_score": 15.0 if true_s1 not in cand_s1s else [c[1] for c in cands if c[0] == true_s1][0],
            "rank": 1
        })
        train_labels.append(1)

    for rank, (cand_s1, sc) in enumerate(cands, start=1):
        if cand_s1 != true_s1:
            train_pairs.append({
                "query_id": qid,
                "s1_id": cand_s1,
                "bm25_score": sc,
                "rank": rank
            })
            train_labels.append(0)

print(f"Total training candidate pairs: {len(train_pairs):,} ({sum(train_labels):,} pos, {len(train_labels) - sum(train_labels):,} neg)")

# 4. Feature Extraction
print("Extracting pairwise discriminator features...")
X_train_df = extract_candidate_features_dataframe(train_pairs, s1_lookup, query_lookup)
y_train = np.array(train_labels, dtype=int)

# 5. Train Asymmetric 4x FP GBDT Ensemble
print(f"Training Asymmetric Precision Ensemble (FP Penalty = {FP_PENALTY_WEIGHT}x)...")
ensemble = AsymmetricPrecisionEnsemble(fp_weight=FP_PENALTY_WEIGHT, n_estimators=450, learning_rate=0.04)
ensemble.fit(X_train_df, y_train)
print("Ensemble training complete!")


## Section 7: Unseen Validation Split Evaluation & Threshold Tuning

We evaluate the ensemble on unseen Validation $S_1$ entities and run a **2D grid search** over decision boundary (`abs_threshold`) and ambiguity rejection (`margin_threshold`) to find the exact configuration targeting **Macro $F_{0.5} \ge 0.99$**.


In [ ]:
print("Evaluating on Unseen Validation S1 partition...")
blocker_val = PrecisionSieveBlocker(top_k_bm25=8)
blocker_val.fit(s1_val_part)

val_qids = [q for q, s1 in q_to_s1_map.items() if s1 in val_s1_set]
sub_val_q_df = queries_train_df[queries_train_df["entity_id"].isin(set(val_qids))]
val_cands_map = blocker_val.generate_candidates(sub_val_q_df)

val_pairs: List[Dict[str, Any]] = []
for row in sub_val_q_df.itertuples():
    qid = row.entity_id
    for rank, (cand_s1, sc) in enumerate(val_cands_map.get(qid, []), start=1):
        val_pairs.append({
            "query_id": qid,
            "s1_id": cand_s1,
            "bm25_score": sc,
            "rank": rank
        })

print(f"Scoring {len(val_pairs):,} validation candidate pairs...")
X_val_df = extract_candidate_features_dataframe(val_pairs, s1_lookup, query_lookup)
val_probs = ensemble.predict_proba(X_val_df, apply_vetoes=True, s1_lookup=s1_lookup, query_lookup=query_lookup)

scored_val = [
    {"query_id": p["query_id"], "s1_id": p["s1_id"], "score": float(pr)}
    for p, pr in zip(val_pairs, val_probs)
]

val_gt_dict = {s1: train_gt_dict.get(s1, set()) for s1 in val_s1_set}

best_abs, best_margin, best_f05, best_metrics = optimize_thresholds_grid(
    scored_candidates=scored_val,
    ground_truth=val_gt_dict,
    all_s1_ids=list(val_s1_set),
    abs_range=[0.55, 0.60, 0.65, 0.70, 0.75, 0.80],
    margin_range=[0.05, 0.08, 0.10, 0.12, 0.15]
)

print("\n=======================================================")
print("             VALIDATION EVALUATION METRICS")
print("=======================================================")
print(f"  Validation Macro F0.5:      {best_metrics.get('macro_f0.5', 0.0):.4f}")
print(f"  Validation Macro Precision: {best_metrics.get('macro_precision', 0.0):.4f}")
print(f"  Validation Macro Recall:    {best_metrics.get('macro_recall', 0.0):.4f}")
print(f"  Optimal Abs Threshold:      {best_abs:.2f}")
print(f"  Optimal Margin Threshold:   {best_margin:.2f}")
print("=======================================================\n")


## Section 8: Memory-Safe Streaming Test Inference Engine

We execute streaming test inference across `test_source1.tsv`, `test_source2.tsv`, and `test_source3.tsv`.
- Tier 1 Exact Sieve locks in clean matches instantly ($O(1)$).
- Tier 2 GBDT Ensemble scores ambiguous candidate pairs with Hard Veto Gates active.
- Disk-sharded streaming keeps peak RAM $< 2.5$ GB and disk usage $< 500$ MB, completely preventing Kaggle out-of-disk crashes.


In [ ]:
match_out = OUTPUT_DIR / "matching_results.tsv"
cand_out = OUTPUT_DIR / "candidate_pairs.tsv"

print(f"Starting streaming inference on {TEST_DIR}...")
infer_result = run_precision_sieve_inference(
    ensemble=ensemble,
    test_dir=TEST_DIR,
    output_matching_path=match_out,
    output_candidate_path=cand_out,
    abs_threshold=best_abs,
    margin_threshold=best_margin,
    chunk_size=50000
)

print("\nStreaming inference completed successfully!")
print(f"  Total Processed Queries: {infer_result['total_queries']:,}")
print(f"  Tier 1 Exact Matches:   {infer_result['tier1_exact_matches']:,}")
print(f"  Total Matched Pairs:    {infer_result['num_matches']:,}")
print(f"  Total Candidate Pairs:  {infer_result['num_candidates']:,}")
print(f"  Execution Time:         {infer_result['runtime_sec']:.1f}s")


## Section 9: Official Hackathon Submission Validation

We invoke the official hackathon validator (`utils/validate_submission.py`) to verify:
1. Required column headers (`source1_entity_id`, `matched_entity_ids`, `candidate_entity_ids`).
2. Exact matching count and singleton representation.
3. Exclusivity: each query matched to at most one reference entity.
4. Invariant: `predicted_pairs ⊆ candidate_pairs`.


In [ ]:
validate_script = PROJECT_ROOT / "utils" / "validate_submission.py"
if validate_script.exists():
    print("Running official submission validator (utils/validate_submission.py)...\n")
    cmd = [
        sys.executable,
        str(validate_script),
        "--matching", str(match_out),
        "--candidate", str(cand_out),
        "--test-dir", str(TEST_DIR)
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if proc.returncode == 0:
        print("\n[SUCCESS] ALL VALIDATION CHECKS PASSED!")
        print(f"Submission files are ready in {OUTPUT_DIR}:")
        print(f"  1. {match_out.name} ({match_out.stat().st_size / (1024**2):.2f} MB)")
        print(f"  2. {cand_out.name} ({cand_out.stat().st_size / (1024**2):.2f} MB)")
    else:
        print(f"[WARNING] Validator returned exit code {proc.returncode}")
else:
    print(f"Validator script not found at {validate_script}")
